In [ ]:
import ast
import os
import glob
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# =========================================================
# CONFIG
# =========================================================
DATA_FOLDER     = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data"
TRAIN_FILE      = os.path.join(DATA_FOLDER, "synthetic_dataset_with_item.csv")
REAL_DATA_FILE  = os.path.join(DATA_FOLDER, "real_participant_data.csv")
MODEL_FILE      = os.path.join(DATA_FOLDER, "reason_prediction_model.pkl")
CONVERTED_FILE  = os.path.join(DATA_FOLDER, "latest_converted_psychopy.csv")
PREDICTION_FILE = os.path.join(DATA_FOLDER, "latest_reason_percentages.csv")
RESULTS_JSON    = os.path.join(DATA_FOLDER, "latest_results.json")

REAL_DATA_THRESHOLD = 10

In [ ]:
# =========================================================
# HELPERS
# =========================================================
def parse_list_cell(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, (int, float)):
        return [value]
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [value]


def get_last_numeric(value):
    nums = [float(v) for v in parse_list_cell(value) if str(v).replace('.','',1).replace('-','',1).isdigit() or isinstance(v, (int,float))]
    try:
        nums = [float(v) for v in parse_list_cell(value)]
    except Exception:
        nums = []
    return nums[-1] if nums else np.nan


def get_last_string(value):
    arr = parse_list_cell(value)
    return str(arr[-1]).strip() if arr else None


def standardize_reason_labels(series):
    return series.astype(str).str.strip().str.lower().replace(
        {"price": "Price", "brand": "Brand", "familiarity": "Familiarity"}
    )


def normalize_gender(value):
    gender_map = {"male": "Male", "m": "Male", "female": "Female", "f": "Female"}
    return gender_map.get(str(value).strip().lower(), str(value).strip().title())


def get_user_age():
    # When run via nbconvert (from Flask), age is passed as NB_AGE env var
    env_age = os.environ.get("NB_AGE", "").strip()
    if env_age:
        try:
            age = int(env_age)
            if age > 0:
                print(f"Age set from environment: {age}")
                return age
        except ValueError:
            pass
    # Fallback: interactive input when running manually
    while True:
        try:
            age = int(input("Enter participant age: ").strip())
            if age > 0:
                return age
            print("Please enter a valid positive age.")
        except ValueError:
            print("Invalid input. Please enter a whole number.")


def get_user_gender():
    # When run via nbconvert (from Flask), gender is passed as NB_GENDER env var
    env_gender = normalize_gender(os.environ.get("NB_GENDER", "").strip())
    if env_gender in ["Male", "Female"]:
        print(f"Gender set from environment: {env_gender}")
        return env_gender
    # Fallback: interactive input when running manually
    while True:
        gender = normalize_gender(input("Enter participant gender (Male/Female): ").strip())
        if gender in ["Male", "Female"]:
            return gender
        print("Invalid input. Please enter Male or Female.")

In [ ]:
# =========================================================
# FIND LATEST PSYCHOPY CSV
# =========================================================
def get_latest_psychopy_csv(folder_path):
    skip = {"synthetic", "converted", "prediction", "latest_reason", "real_participant"}
    valid = [
        f for f in glob.glob(os.path.join(folder_path, "*.csv"))
        if not any(kw in os.path.basename(f).lower() for kw in skip)
    ]
    if not valid:
        raise FileNotFoundError("No valid PsychoPy CSV files found.")
    return max(valid, key=os.path.getmtime)

In [ ]:
# =========================================================
# TRAIN MODEL
# Fix 1: brochure_id → categorical (one-hot)
# Fix 2: clicked_item removed (was data leakage)
# Fix 3: 5-fold CV for reliable accuracy
# Fix 4: uses real data when >= threshold
# =========================================================
def train_model(train_file, real_data_file, model_file):
    use_real = False
    if os.path.exists(real_data_file):
        real_df = pd.read_csv(real_data_file)
        labeled = real_df["reason"].notna().sum() if "reason" in real_df.columns else 0
        if labeled >= REAL_DATA_THRESHOLD:
            df = real_df.copy()
            use_real = True
            print(f"Training on REAL participant data ({len(df)} rows).")

    if not use_real:
        df = pd.read_csv(train_file)
        print(f"Training on SYNTHETIC data ({len(df)} rows). "
              f"Label {REAL_DATA_THRESHOLD} real rows to switch automatically.")

    df = df.copy()
    df["reason"]      = standardize_reason_labels(df["reason"])
    df["gender"]      = df["gender"].astype(str).str.strip().str.title()
    df["brochure_id"] = df["brochure_id"].astype(str)          # categorical
    df = df.dropna(subset=["reason"])
    df = df[df["reason"].isin(["Brand", "Price", "Familiarity"])].reset_index(drop=True)

    # Features — no clicked_item
    feature_cols      = ["brochure_id", "age", "gender", "reaction_time", "gaze_x", "gaze_y"]
    numeric_features  = ["age", "reaction_time", "gaze_x", "gaze_y"]
    categorical_features = ["gender", "brochure_id"]

    X = df[feature_cols].copy()
    y = df["reason"].copy()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer,     numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300, max_depth=8,
            min_samples_leaf=2, random_state=42,
            class_weight="balanced"
        ))
    ])

    # 5-fold cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    print(f"\n5-Fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    # Final fit on 80/20 split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)

    print("\n======================================")
    print("MODEL TRAINING RESULTS")
    print("======================================")
    print(f"Test Accuracy : {round(test_acc, 4)}")
    print(f"CV Accuracy   : {round(cv_scores.mean(), 4)} ± {round(cv_scores.std(), 4)}")
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    joblib.dump(model, model_file)
    print(f"Model saved to: {model_file}\n")
    return model, test_acc, cv_scores

In [ ]:
# =========================================================
# CONVERT RAW PSYCHOPY FILE TO TIDY FORMAT
# =========================================================
def convert_psychopy_to_tidy(psychopy_file, age_value, gender_value, output_file):
    raw_df = pd.read_csv(psychopy_file)
    if raw_df.empty:
        raise ValueError("The PsychoPy CSV file is empty.")

    participant_value = "Unknown"
    if "participant" in raw_df.columns:
        val = raw_df["participant"].dropna()
        if not val.empty:
            participant_value = str(val.iloc[0]).strip()

    brochure_blocks = [
        {"brochure_id": 1, "time_col": "mouse_5.time", "x_col": "mouse_5.x", "y_col": "mouse_5.y", "clicked_col": "mouse_5.clicked_name"},
        {"brochure_id": 2, "time_col": "mouse_4.time", "x_col": "mouse_4.x", "y_col": "mouse_4.y", "clicked_col": "mouse_4.clicked_name"},
        {"brochure_id": 3, "time_col": "mouse_3.time", "x_col": "mouse_3.x", "y_col": "mouse_3.y", "clicked_col": "mouse_3.clicked_name"},
        {"brochure_id": 4, "time_col": "mouse.time",   "x_col": "mouse.x",   "y_col": "mouse.y",   "clicked_col": "mouse.clicked_name"},
        {"brochure_id": 5, "time_col": "mouse_2.time", "x_col": "mouse_2.x", "y_col": "mouse_2.y", "clicked_col": "mouse_2.clicked_name"},
    ]

    tidy_rows = []
    for block in brochure_blocks:
        bid   = block["brochure_id"]
        c_col = block["clicked_col"]
        if c_col not in raw_df.columns:
            print(f"Skipping brochure {bid}: missing column {c_col}")
            continue
        valid_rows = raw_df[raw_df[c_col].notna()]
        if valid_rows.empty:
            print(f"Skipping brochure {bid}: no clicked item found")
            continue
        row = valid_rows.iloc[0]
        tidy_rows.append({
            "participant":   participant_value,
            "brochure_id":   str(bid),
            "age":           age_value,
            "gender":        gender_value,
            "reaction_time": get_last_numeric(row[block["time_col"]]) if block["time_col"] in raw_df.columns else np.nan,
            "gaze_x":        get_last_numeric(row[block["x_col"]])    if block["x_col"]   in raw_df.columns else np.nan,
            "gaze_y":        get_last_numeric(row[block["y_col"]])    if block["y_col"]   in raw_df.columns else np.nan,
            "clicked_item":  get_last_string(row[c_col])
        })

    tidy_df = pd.DataFrame(tidy_rows).sort_values("brochure_id").reset_index(drop=True)
    if tidy_df.empty:
        raise ValueError("No valid brochure data extracted.")

    tidy_df.to_csv(output_file, index=False)
    print("======================================")
    print("CONVERTED LATEST PSYCHOPY FILE")
    print("======================================")
    print(tidy_df.to_string(index=False))
    print(f"\nSaved to: {output_file}\n")
    return tidy_df

In [ ]:
# =========================================================
# ACCUMULATE REAL PARTICIPANT DATA
# After each session, open real_participant_data.csv in Excel
# and fill in the 'reason' column (Brand / Price / Familiarity)
# =========================================================
def accumulate_real_data(tidy_df, real_data_file):
    df = tidy_df.copy()
    df["reason"] = np.nan   # label manually in CSV after each session

    if os.path.exists(real_data_file):
        existing = pd.read_csv(real_data_file)
        combined = pd.concat([existing, df], ignore_index=True)
    else:
        combined = df

    combined.to_csv(real_data_file, index=False)
    labeled = int(combined["reason"].notna().sum())
    total   = len(combined)
    print(f"Real data updated: {total} rows total, {labeled} labeled.")
    if labeled < REAL_DATA_THRESHOLD:
        print(f"  → Label {REAL_DATA_THRESHOLD - labeled} more rows in real_participant_data.csv to use real data for training.")
    else:
        print(f"  → Real data threshold reached! Next training will use real data.")

In [ ]:
# =========================================================
# PREDICT — Brand / Familiarity / Price percentages
# No SHAP, no XAI — just the three influence percentages
# =========================================================
def predict_reason_percentages(tidy_df, model_file, prediction_file, results_json_file):
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model not found: {model_file}")

    model = joblib.load(model_file)

    feature_cols = ["brochure_id", "age", "gender", "reaction_time", "gaze_x", "gaze_y"]
    X_new = tidy_df[feature_cols].copy()
    X_new["brochure_id"] = X_new["brochure_id"].astype(str)

    probs       = model.predict_proba(X_new)
    preds       = model.predict(X_new)
    class_names = model.named_steps["classifier"].classes_

    prob_df = pd.DataFrame(probs * 100, columns=[f"{c}_Percentage" for c in class_names]).round(2)

    final_df = tidy_df.copy()
    final_df["Predicted_Reason"] = preds
    for col in prob_df.columns:
        final_df[col] = prob_df[col]

    final_df.to_csv(prediction_file, index=False)

    # ── Print report ──────────────────────────────────────
    print("======================================")
    print("REASON INFLUENCE REPORT")
    print("======================================\n")

    results_rows = []
    for _, row in final_df.iterrows():
        brand_pct = float(row.get("Brand_Percentage", 0))
        fam_pct   = float(row.get("Familiarity_Percentage", 0))
        price_pct = float(row.get("Price_Percentage", 0))
        rt        = row["reaction_time"]

        print(f"Participant   : {row['participant']}")
        print(f"Brochure ID   : {row['brochure_id']}")
        print(f"Clicked Item  : {row.get('clicked_item', 'N/A')}")
        print(f"Age / Gender  : {row['age']} / {row['gender']}")
        print(f"Reaction Time : {rt:.4f}s" if pd.notna(rt) else "Reaction Time : NaN")
        print(f"\nReason influence percentages:")
        print(f"   Brand       : {brand_pct:.2f}%")
        print(f"   Familiarity : {fam_pct:.2f}%")
        print(f"   Price       : {price_pct:.2f}%")
        print(f"\n→ Final Decision Most Influenced By : {row['Predicted_Reason']}")
        print("--------------------------------------\n")

        results_rows.append({
            "participant":     str(row["participant"]),
            "brochure_id":     row["brochure_id"],
            "clicked_item":    str(row.get("clicked_item", "N/A")),
            "age":             int(row["age"]),
            "gender":          str(row["gender"]),
            "reaction_time":   round(float(rt), 4) if pd.notna(rt) else None,
            "gaze_x":          round(float(row["gaze_x"]), 4) if pd.notna(row["gaze_x"]) else None,
            "gaze_y":          round(float(row["gaze_y"]), 4) if pd.notna(row["gaze_y"]) else None,
            "brand_pct":       round(brand_pct, 2),
            "familiarity_pct": round(fam_pct, 2),
            "price_pct":       round(price_pct, 2),
            "predicted_reason": str(row["Predicted_Reason"])
        })

    # ── Write JSON for the dashboard ──────────────────────
    with open(results_json_file, "w") as f:
        json.dump({"predictions": results_rows}, f, indent=2)

    print(f"Prediction CSV : {prediction_file}")
    print(f"Results JSON   : {results_json_file}\n")
    return final_df

In [ ]:
# =========================================================
# MAIN
# =========================================================
# RUN_FROM_FLASK is set by app.py when executing via nbconvert.
# This prevents the block from running on accidental cell re-runs in Jupyter.
import os as _os
_run_mode = _os.environ.get("RUN_FROM_FLASK", "") or _os.environ.get("NB_AGE", "")

if _run_mode or __name__ == "__main__":
    age_value    = get_user_age()
    gender_value = get_user_gender()

    train_model(TRAIN_FILE, REAL_DATA_FILE, MODEL_FILE)

    latest_csv = get_latest_psychopy_csv(DATA_FOLDER)
    print(f"Latest PsychoPy CSV: {latest_csv}\n")

    tidy_df = convert_psychopy_to_tidy(latest_csv, age_value, gender_value, CONVERTED_FILE)

    accumulate_real_data(tidy_df, REAL_DATA_FILE)

    predict_reason_percentages(tidy_df, MODEL_FILE, PREDICTION_FILE, RESULTS_JSON)